In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # must be set before importing jax


In [2]:
def _to_str(x):
    return x.decode() if isinstance(x, (bytes, bytearray)) else str(x)


import pynvml
pynvml.nvmlInit()

def current_gpu_status(gpu_index=0):
    h = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
    name = _to_str(pynvml.nvmlDeviceGetName(h))
    mem = pynvml.nvmlDeviceGetMemoryInfo(h)
    util = pynvml.nvmlDeviceGetUtilizationRates(h)
    return {
        "gpu": gpu_index,
        "name": name,
        "memory_used_mb": mem.used / 1024**2,
        "memory_total_mb": mem.total / 1024**2,
        "memory_free_mb": mem.free / 1024**2,
        "utilization_gpu_pct": util.gpu,
        "utilization_mem_pct": util.memory,
    }

current_gpu_status()


{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 767.25,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 81152.75,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [3]:
import jax.numpy as jnp
(jnp.ones((1,), dtype=jnp.float32) + 1).block_until_ready()
current_gpu_status()


{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 1200.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 80719.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [4]:
1

1

In [5]:
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", SettingWithCopyWarning)

import numpy as np
import pandas as pd
import seaborn as sns
import jax
import functools
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import rapids_singlecell as rsc
import flax.linen as nn
import optax
import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear
from cellflow.plotting import plot_condition_embedding
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca, reconstruct_pca
from cellflow.metrics import compute_r_squared, compute_e_distance


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module


In [6]:
current_gpu_status()

{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 1200.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 80719.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [12]:
adata = cellflow.datasets.pbmc_cytokines()

In [13]:
adata.obs["condition"] = adata.obs.apply(lambda x: x["donor"] + "_" + x["cytokine"], axis=1)
adata.obs["is_control"] = adata.obs.apply(lambda x: True if x["cytokine"]=="PBS" else False, axis=1)

In [14]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [15]:
small_adata_for_pca = sc.pp.subsample(adata, n_obs=100, copy=True)


In [16]:
cfpp.centered_pca(small_adata_for_pca, n_comps=100, method="scanpy", keep_centered_data=False)
cfpp.project_pca(query_adata=adata, ref_adata=small_adata_for_pca)

In [17]:
current_gpu_status()


{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 1200.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 80719.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [19]:
cf = CellFlow(adata, solver="otfm")

In [20]:
current_gpu_status()


{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 1200.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 80719.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [21]:
cf.prepare_data(
    sample_rep = "X_pca",
    control_key = "is_control",
    perturbation_covariates = {"cytokine_treatment": ("cytokine",)},
    perturbation_covariate_reps = {"cytokine_treatment": "esm2_embeddings"},
    sample_covariates = ["donor"],
    sample_covariate_reps = {"donor": "donor_embeddings"},
    split_covariates = ["donor"],
    max_combination_length = 1,
    null_value = 0.0,
)

[########################################] | 100% Completed | 114.83 ms
[########################################] | 100% Completed | 2.63 sms
[########################################] | 100% Completed | 208.24 ms


In [22]:
current_gpu_status()

{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 1200.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 80719.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [23]:
layers_before_pool = {
    "cytokine_treatment": {"layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.5},
    "donor": {"layer_type": "mlp", "dims": [256, 256], "dropout_rate": 0.0},
}

layers_after_pool = {
    "layer_type": "mlp", "dims": [1024, 1024], "dropout_rate": 0.0,
}

In [24]:
match_fn = functools.partial(match_linear, epsilon=0.5, tau_a=1.0, tau_b=1.0)

In [25]:
cf.prepare_model(
    condition_mode="deterministic",
    regularization=0.0,
    pooling="attention_token",
    pooling_kwargs={},
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.9,
    condition_encoder_kwargs={},
    pool_sample_covariates=True,
    time_freqs=1024,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="concatenation",
    decoder_dims=[4096, 4096, 4096],
    vf_act_fn=nn.silu,
    vf_kwargs=None,
    probability_path={"constant_noise": 0.5},
    match_fn=match_fn,
    optimizer=optax.MultiSteps(optax.adam(5e-5), 20),
    solver_kwargs={},
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
)

In [26]:
current_gpu_status()

{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 3266.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 78653.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [27]:
callbacks = []

In [28]:
cf.train(
    num_iterations=2,
    batch_size=1024,
    callbacks=callbacks,
    valid_freq=30_000,
)

100%|██████████| 2/2 [00:11<00:00,  5.80s/it]


In [29]:
current_gpu_status()

{'gpu': 0,
 'name': 'NVIDIA A100 80GB PCIe',
 'memory_used_mb': 5318.0625,
 'memory_total_mb': 81920.0,
 'memory_free_mb': 76601.9375,
 'utilization_gpu_pct': 0,
 'utilization_mem_pct': 0}

In [ ]:
cf.solver.step_fn()

In [30]:
import time, threading
import pandas as pd
import pynvml
pynvml.nvmlInit()

def _to_str(x):
    return x.decode() if isinstance(x, (bytes, bytearray)) else str(x)

def nvml_handle(gpu_index=0):
    return pynvml.nvmlDeviceGetHandleByIndex(gpu_index)

def nvml_used_mb(h):
    return pynvml.nvmlDeviceGetMemoryInfo(h).used / 1024**2

class NVMLPeakSampler:
    def __init__(self, gpu_index=0, interval_s=0.01):
        self.h = nvml_handle(gpu_index)
        self.interval_s = interval_s
        self.samples = []
        self._stop = threading.Event()
        self._t = None

    def start(self):
        self.samples = []
        self._stop.clear()
        def loop():
            while not self._stop.is_set():
                self.samples.append((time.perf_counter(), nvml_used_mb(self.h)))
                time.sleep(self.interval_s)
        self._t = threading.Thread(target=loop, daemon=True)
        self._t.start()

    def stop(self):
        self._stop.set()
        if self._t is not None:
            self._t.join()

    def peak_mb(self):
        return max(m for _, m in self.samples) if self.samples else float("nan")

    def df(self):
        return pd.DataFrame(self.samples, columns=["t", "used_mb"])


In [31]:
import jax

def benchmark_peak_gpu_mem(step_fn, cf_sampler, *, warmup=3, iters=10, gpu_index=0, sample_interval_s=0.01, label="run"):
    h = nvml_handle(gpu_index)
    rng_jax = jax.random.PRNGKey(0)
    rng_np = np.random.default_rng(0)

    
    for _ in range(warmup):
        rng_jax, rng_step_fn = jax.random.split(rng_jax, 2)
        batch = cf_sampler.sample(rng_np)
        loss = step_fn(rng_step_fn, batch)
        jax.block_until_ready(loss)

    baseline_mb = nvml_used_mb(h)

    # Measure peak during multiple steady-state iterations
    sampler = NVMLPeakSampler(gpu_index=gpu_index, interval_s=sample_interval_s)
    sampler.start()
    t0 = time.perf_counter()
    for _ in range(iters):
        rng_jax, rng_step_fn = jax.random.split(rng_jax, 2)
        batch = cf_sampler.sample(rng_np)
        loss = step_fn(rng_step_fn, batch)
        jax.block_until_ready(loss)
    t1 = time.perf_counter()
    sampler.stop()

    peak_mb = sampler.peak_mb()
    inc_peak_mb = peak_mb - baseline_mb

    summary = pd.DataFrame([{
        "label": label,
        "baseline_used_mb": baseline_mb,
        "peak_used_mb": peak_mb,
        "incremental_peak_mb": inc_peak_mb,
        "iters": iters,
        "sample_interval_ms": sample_interval_s * 1000,
        "elapsed_s": t1 - t0,
        "per_iter_ms": (t1 - t0) * 1000 / iters,
    }])

    return summary, sampler.df()


In [32]:
summary, trace = benchmark_peak_gpu_mem(cf.solver.step_fn, cf.dataloader, warmup=3, iters=100, sample_interval_s=0.01, label="example")

In [33]:
summary

,label,baseline_used_mb,peak_used_mb,incremental_peak_mb,iters,sample_interval_ms,elapsed_s,per_iter_ms
0,example,5318.0625,5318.0625,0.0,100,10.0,1.657742,16.577416


In [36]:
cf.dataloader.batch_size = 10000

In [ ]:
summary, trace = benchmark_peak_gpu_mem(cf.solver.step_fn, cf.dataloader, warmup=3, iters=100, sample_interval_s=0.01, label="example")

In [ ]:
summary

In [20]:
cf.dataloader

In [1]:

import time
import pandas as pd

import jax
import jax.numpy as jnp

# --- NVML setup (GPU memory queries) ---
try:
    import pynvml
    pynvml.nvmlInit()
    _NVML_OK = True
except Exception as e:
    _NVML_OK = False
    raise RuntimeError(
        "NVML not available. Install with `pip install pynvml` and ensure you're on an NVIDIA GPU."
    ) from e

def _get_gpu_handle(gpu_index: int = 0):
    return pynvml.nvmlDeviceGetHandleByIndex(gpu_index)

def gpu_mem_used_mb(handle) -> float:
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.used / (1024**2)

def gpu_mem_total_mb(handle) -> float:
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.total / (1024**2)

# --- Define your JAX "step" here ---
# Replace this with your training step / forward pass.
@jax.jit
def step(x):
    y = jnp.sin(x) @ jnp.cos(x)
    return y.sum()

# --- Benchmark harness ---
def benchmark_jax_step(
    step_fn,
    inputs,
    *,
    warmup: int = 3,
    iters: int = 20,
    gpu_index: int = 0,
    label: str = "run1",
):
    """
    Runs warmup steps (compile + cache), then benchmarks steady-state.
    Returns a pandas DataFrame with per-iteration timings and memory.
    """
    handle = _get_gpu_handle(gpu_index)
    total_mb = gpu_mem_total_mb(handle)

    # Ensure inputs are on device (optional, but reduces host->device noise)
    inputs_dev = jax.device_put(inputs)

    # Warmup (includes compilation on first call)
    for _ in range(warmup):
        out = step_fn(inputs_dev)
        out.block_until_ready()

    rows = []
    # Baseline memory before measured loop
    base_mem = gpu_mem_used_mb(handle)

    for i in range(iters):
        mem_before = gpu_mem_used_mb(handle)
        t0 = time.perf_counter()

        out = step_fn(inputs_dev)
        out.block_until_ready()

        t1 = time.perf_counter()
        mem_after = gpu_mem_used_mb(handle)

        rows.append(
            {
                "label": label,
                "iter": i,
                "time_ms": (t1 - t0) * 1000.0,
                "gpu_mem_before_mb": mem_before,
                "gpu_mem_after_mb": mem_after,
                "gpu_mem_delta_mb": mem_after - mem_before,
                "gpu_mem_total_mb": total_mb,
            }
        )

    df = pd.DataFrame(rows)

    # Add summary columns (same value repeated for convenience)
    df["gpu_mem_baseline_mb"] = base_mem
    df["gpu_mem_peak_mb"] = df[["gpu_mem_before_mb", "gpu_mem_after_mb"]].max(axis=1).cummax()
    df["time_ms_mean"] = df["time_ms"].mean()
    df["time_ms_p50"] = df["time_ms"].median()
    df["time_ms_p95"] = df["time_ms"].quantile(0.95)
    df["gpu_mem_peak_overall_mb"] = df["gpu_mem_peak_mb"].max()
    df["gpu_mem_peak_overall_pct"] = 100.0 * df["gpu_mem_peak_overall_mb"] / total_mb

    return df

# --- Example inputs (adjust shapes/dtypes to match your code) ---
x = jnp.ones((4096, 4096), dtype=jnp.float32)

df = benchmark_jax_step(step, x, warmup=3, iters=30, gpu_index=0, label="sinmatmul")

df.head()


,label,iter,time_ms,gpu_mem_before_mb,gpu_mem_after_mb,gpu_mem_delta_mb,gpu_mem_total_mb,gpu_mem_baseline_mb,gpu_mem_peak_mb,time_ms_mean,time_ms_p50,time_ms_p95,gpu_mem_peak_overall_mb,gpu_mem_peak_overall_pct
0,sinmatmul,0,1.307208,62078.0625,62078.0625,0.0,81920.0,62078.0625,62078.0625,1.292936,1.290206,1.32187,62078.0625,75.778885
1,sinmatmul,1,1.314189,62078.0625,62078.0625,0.0,81920.0,62078.0625,62078.0625,1.292936,1.290206,1.32187,62078.0625,75.778885
2,sinmatmul,2,1.298113,62078.0625,62078.0625,0.0,81920.0,62078.0625,62078.0625,1.292936,1.290206,1.32187,62078.0625,75.778885
3,sinmatmul,3,1.283523,62078.0625,62078.0625,0.0,81920.0,62078.0625,62078.0625,1.292936,1.290206,1.32187,62078.0625,75.778885
4,sinmatmul,4,1.294497,62078.0625,62078.0625,0.0,81920.0,62078.0625,62078.0625,1.292936,1.290206,1.32187,62078.0625,75.778885
